# Batch Processing and Performance

This notebook demonstrates efficient batch processing techniques for large datasets and provides performance optimization tips.

## Setup

Load libraries and create a larger sample dataset for demonstration.

In [ ]:
import pandas as pd
import ethnicolr
import time
import numpy as np
from pathlib import Path

# Load and replicate sample data to create a larger dataset
data_path = Path('data/input-with-header.csv')
small_df = pd.read_csv(data_path)
print(f"Original sample size: {len(small_df)}")

# Create a larger dataset by replicating and adding some variation
large_df = pd.concat([small_df] * 10, ignore_index=True)
print(f"Larger dataset size: {len(large_df)}")

# Preview the data
large_df.head()

## Performance Comparison

Let's compare the performance of different models on our dataset.

In [ ]:
def time_prediction(func, df, *args, **kwargs):
    """Helper function to time predictions"""
    start_time = time.time()
    result = func(df, *args, **kwargs)
    end_time = time.time()
    return result, end_time - start_time

# Test different models
models = {
    'census_lookup': (ethnicolr.census_ln, ['last_name'], {'year': 2010}),
    'census_lstm': (ethnicolr.pred_census_ln, ['last_name'], {}),
    'wiki_lastname': (ethnicolr.pred_wiki_ln, ['last_name'], {}),
    'florida_lstm': (ethnicolr.pred_fl_reg_ln, ['last_name'], {})
}

performance_results = {}

for model_name, (func, args, kwargs) in models.items():
    print(f"\nTesting {model_name}...")
    result, duration = time_prediction(func, large_df, *args, **kwargs)
    
    performance_results[model_name] = {
        'duration': duration,
        'rows_per_second': len(large_df) / duration,
        'result_shape': result.shape
    }
    
    print(f"Duration: {duration:.2f} seconds")
    print(f"Speed: {len(large_df) / duration:.0f} rows/second")

# Performance summary
perf_df = pd.DataFrame(performance_results).T

# Convert numeric columns explicitly to avoid TypeError
perf_df['duration'] = pd.to_numeric(perf_df['duration'], errors='coerce').round(2)
perf_df['rows_per_second'] = pd.to_numeric(perf_df['rows_per_second'], errors='coerce').round(0)

print("\nPerformance Summary:")
perf_df[['duration', 'rows_per_second']]

## Chunked Processing

For very large datasets, processing in chunks can be more memory efficient.

In [ ]:
def process_in_chunks(df, func, chunk_size=1000, *args, **kwargs):
    """Process dataframe in chunks to manage memory usage"""
    results = []
    total_chunks = (len(df) - 1) // chunk_size + 1
    
    for i in range(0, len(df), chunk_size):
        chunk = df.iloc[i:i + chunk_size]
        chunk_result = func(chunk, *args, **kwargs)
        results.append(chunk_result)
        
        if (i // chunk_size + 1) % 5 == 0:  # Progress every 5 chunks
            print(f"Processed {i // chunk_size + 1}/{total_chunks} chunks")
    
    return pd.concat(results, ignore_index=True)

# Example: Process in chunks of 250 rows
print("Processing Florida model in chunks of 250...")
start_time = time.time()
chunked_result = process_in_chunks(
    large_df, 
    ethnicolr.pred_fl_reg_ln, 
    250,
    'last_name'
)
chunked_duration = time.time() - start_time

print(f"\nChunked processing completed in {chunked_duration:.2f} seconds")
print(f"Result shape: {chunked_result.shape}")
chunked_result[['last_name', 'race', 'nh_white', 'nh_black', 'asian', 'hispanic']].head()

## Handling Missing or Problematic Names

Real-world datasets often have missing values, special characters, or other data quality issues.

In [ ]:
# Create a dataset with some problematic entries
problematic_df = large_df.copy().head(50)

# Add some missing values and problematic names
problematic_df.loc[5, 'last_name'] = None
problematic_df.loc[10, 'last_name'] = ''
problematic_df.loc[15, 'last_name'] = 'O\'Connor'  # Apostrophe
problematic_df.loc[20, 'last_name'] = 'García'     # Accented character
problematic_df.loc[25, 'last_name'] = '123'        # Numeric
problematic_df.loc[30, 'first_name'] = None

print("Sample problematic entries:")
print(problematic_df.iloc[[5, 10, 15, 20, 25, 30]][['first_name', 'last_name']])

# Process with Wikipedia model (handles problematic names better)
wiki_result = ethnicolr.pred_wiki_name(problematic_df, 'last_name', 'first_name')

print("\nProcessing results for problematic names:")
problem_indices = [5, 10, 15, 20, 25, 30]
display_cols = ['first_name', 'last_name', 'race', '__name', 'processing_status']
# Some columns might not exist, so filter to available ones
available_cols = [col for col in display_cols if col in wiki_result.columns]
print(wiki_result.iloc[problem_indices][available_cols])

## Data Quality Analysis

Analyze the quality and coverage of predictions across your dataset.

In [ ]:
# Get predictions for quality analysis
census_pred = ethnicolr.pred_census_ln(large_df, 'last_name')
wiki_pred = ethnicolr.pred_wiki_ln(large_df, 'last_name')

# Calculate prediction confidence for different model types
# Census models have simple probability columns
census_prob_cols = ['white', 'black', 'api', 'hispanic']
census_pred['max_confidence'] = census_pred[census_prob_cols].max(axis=1)

# Wikipedia models have detailed ethnic category columns - calculate max confidence differently
wiki_prob_cols = [col for col in wiki_pred.columns if col.startswith(('Asian,', 'GreaterAfrican,', 'GreaterEuropean,'))]
wiki_pred['max_confidence'] = wiki_pred[wiki_prob_cols].max(axis=1)

# Confidence distribution
print("Census Model Confidence Distribution:")
print(f"High confidence (>0.8): {(census_pred['max_confidence'] > 0.8).sum()} ({(census_pred['max_confidence'] > 0.8).mean()*100:.1f}%)")
print(f"Medium confidence (0.5-0.8): {((census_pred['max_confidence'] > 0.5) & (census_pred['max_confidence'] <= 0.8)).sum()} ({((census_pred['max_confidence'] > 0.5) & (census_pred['max_confidence'] <= 0.8)).mean()*100:.1f}%)")
print(f"Low confidence (<0.5): {(census_pred['max_confidence'] <= 0.5).sum()} ({(census_pred['max_confidence'] <= 0.5).mean()*100:.1f}%)")

print("\nWikipedia Model Confidence Distribution:")
print(f"High confidence (>0.8): {(wiki_pred['max_confidence'] > 0.8).sum()} ({(wiki_pred['max_confidence'] > 0.8).mean()*100:.1f}%)")
print(f"Medium confidence (0.5-0.8): {((wiki_pred['max_confidence'] > 0.5) & (wiki_pred['max_confidence'] <= 0.8)).sum()} ({((wiki_pred['max_confidence'] > 0.5) & (wiki_pred['max_confidence'] <= 0.8)).mean()*100:.1f}%)")
print(f"Low confidence (<0.5): {(wiki_pred['max_confidence'] <= 0.5).sum()} ({(wiki_pred['max_confidence'] <= 0.5).mean()*100:.1f}%)")

## Batch Processing Best Practices

### Performance Tips:
1. **Choose the right model**: Census lookup is fastest, ML models are slower but more accurate
2. **Use chunking**: For datasets >10,000 rows, process in chunks to manage memory
3. **Clean data first**: Remove/handle missing values before processing
4. **Monitor confidence**: Low confidence predictions may need manual review

### Memory Management:
- Process in chunks of 500-2000 rows for large datasets
- Use only the columns you need in your input DataFrame
- Clear intermediate results when not needed

### Error Handling:
- Check for missing values in name columns
- Handle special characters and accents
- Validate results and flag low-confidence predictions

In [ ]:
# Define helper function for chunked processing
def process_in_chunks(df, func, chunk_size=1000, *args, **kwargs):
    """Process dataframe in chunks to manage memory usage"""
    results = []
    total_chunks = (len(df) - 1) // chunk_size + 1
    
    for i in range(0, len(df), chunk_size):
        chunk = df.iloc[i:i + chunk_size]
        chunk_result = func(chunk, *args, **kwargs)
        results.append(chunk_result)
        
        if (i // chunk_size + 1) % 5 == 0:  # Progress every 5 chunks
            print(f"Processed {i // chunk_size + 1}/{total_chunks} chunks")
    
    return pd.concat(results, ignore_index=True)

# Example production-ready batch processing function
def robust_batch_predict(df, name_col, model='census', chunk_size=1000, min_confidence=0.5):
    """
    Robust batch prediction with error handling and quality filtering
    """
    # Data validation
    if name_col not in df.columns:
        raise ValueError(f"Column '{name_col}' not found in DataFrame")
    
    # Clean data
    clean_df = df.copy()
    clean_df[name_col] = clean_df[name_col].fillna('').astype(str)
    
    # Choose prediction function and define probability columns
    if model == 'census':
        pred_func = ethnicolr.pred_census_ln
        prob_cols = ['white', 'black', 'api', 'hispanic']
    elif model == 'wiki':
        pred_func = ethnicolr.pred_wiki_ln
        # Wikipedia models have detailed ethnic columns
        prob_cols = None  # Will be determined after prediction
    elif model == 'florida':
        pred_func = ethnicolr.pred_fl_reg_ln
        prob_cols = ['nh_white', 'nh_black', 'asian', 'hispanic']
    else:
        raise ValueError(f"Unknown model: {model}")
    
    # Process in chunks
    result = process_in_chunks(clean_df, pred_func, chunk_size, name_col)
    
    # Calculate confidence and add quality flags
    if model == 'wiki':
        # For Wikipedia models, find ethnic category columns
        prob_cols = [col for col in result.columns if col.startswith(('Asian,', 'GreaterAfrican,', 'GreaterEuropean,'))]
    
    result['max_confidence'] = result[prob_cols].max(axis=1)
    result['high_confidence'] = result['max_confidence'] >= min_confidence
    
    return result

# Example usage
print("Running robust batch prediction...")
robust_result = robust_batch_predict(
    large_df.head(100), 
    'last_name', 
    model='census', 
    chunk_size=50,
    min_confidence=0.6
)

print(f"\nProcessed {len(robust_result)} names")
print(f"High confidence predictions: {robust_result['high_confidence'].sum()} ({robust_result['high_confidence'].mean()*100:.1f}%)")
print("\nSample results:")
robust_result[['last_name', 'race', 'max_confidence', 'high_confidence']].head(10)